# Extra BARC labels (2017-2020): what they are and how far to trust them

This notebook re-runs the checks in `classifier/labels/` from the published sources and adds one new check
(outline overlap). Run it from `classifier/labels/`.

## Findings

1. **What the labels are.** The challenge gives 44 labeled field-years for 2021-2023. The Prithvi training table
   (`classifier/prithvi/field_years.csv`) adds 83 field-years from 2017-2020, taken from the farm's own records
   (Zenodo 8087301). These are earlier years, not newer data. The wider label table also holds 2014-2016 and 2024,
   which are not used for training.
2. **Agreement with the official labels.** Reproduced, no class conflicts except one mixed harvest. But the official
   labels only exist for 2021-2023, so the 2017-2020 labels are confirmed indirectly: the same source agrees with the
   official labels in 2021, and the two record sources agree with each other on 47 field-years in 2017-2020.
3. **A planting record is a plan.** Silage planned at planting was harvested as grain (wholly or partly) in 1 of 17
   field-years in Zenodo and 3 of 25 in the farm reports. Planned grain never changed. 56 of the 83 training labels
   rest on a planting record only, 12 of them Silage.
4. **Outline fit.** 10 of the 83 training field-years (all Grain) use an outline that does not fit the field in that
   year. The other 73 are confirmed by overlap with a second outline, by recorded acres, or both.
5. **Difference from the code on `main`.** It flags the same 10 field-years but still trains on all 83. This branch
   adds `outline_check.py`, two columns in `field_years.csv` and `prithvi_train.py --exclude`, so the 10 and the 12
   planting-only Silage labels can be left out. No model has been retrained without them yet.

## 0. Setup and download

In [1]:
import io, subprocess, sys, zipfile, urllib.request
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely import wkt

HERE = Path.cwd()                                   # classifier/labels
ROOT = next(p for p in [HERE, *HERE.parents] if (p / "04_Evaluation").exists())
LABELS = ROOT / "classifier" / "labels"
GPKG = ROOT / "04_Evaluation" / "BARC_Corn.2021-2023.gpkg"
SRC = ROOT.parent / "labels_src"                    # downloads, kept outside the repository
WORK = SRC / "rerun"                                # re-run outputs, so results/ in the repository is not touched
(WORK / "txt").mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200)


def run(script, *args, tail=None, quiet=False):
    r = subprocess.run([sys.executable, str(LABELS / script), *map(str, args)], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stderr)
    out = r.stdout.replace(str(ROOT.parent), "..").rstrip().splitlines()
    if not quiet:
        print("\n".join(out[-tail:] if tail else out))


run("get_sources.py", SRC)

farm_database_annual_planting_harvesting_after_some_screening..zip: md5 65590c4f15b2179599bf676172ab36ba matches the published checksum, 16,712 bytes, 8 files -> zenodo/farmdb
barc_field_outlines_for_study.zip: md5 35fae5dc89622d483b11e4a370f26d9c matches the published checksum, 119,938 bytes, 4 files -> zenodo/outlines
BARCFarmMgmt.zip: md5 1d1d01cf82d81afa2006ea5e342b28d3 matches the published checksum, 11,261,167 bytes, 49 files -> farm_reports/pdf


## 1. Which years do the labels cover?

`extra_labels_outside_2021_2023.csv` is every farm-record label outside the evaluation years. `field_years.csv` is
what the Prithvi model is trained on.

In [2]:
extra = pd.read_csv(LABELS / "results" / "extra_labels_outside_2021_2023.csv")
print("All farm-record labels outside 2021-2023:", len(extra), "field-years")
print(pd.crosstab(extra.year, extra.sources, margins=True).to_string())

fy = pd.read_csv(ROOT / "classifier" / "prithvi" / "field_years.csv")
fy["idn"] = fy.farm.str.replace(" ", "") + "_" + fy.field.astype(str)
print("\nTraining table:", len(fy), "field-years")
print(pd.crosstab(fy.year, fy.label_source, margins=True).to_string())

All farm-record labels outside 2021-2023: 203 field-years
sources  reports  reports+zenodo  zenodo  All
year                                         
2014          17               0       0   17
2015          18               0       0   18
2016          13               0       0   13
2017           3              10      44   57
2018           1              10      13   24
2019           2              16      17   35
2020           1              11      14   26
2024          13               0       0   13
All           68              47      88  203

Training table: 127 field-years
label_source  evaluation labels 2021-2023  farm records 2017-2020  All
year                                                                  
2017                                    0                      24   24
2018                                    0                      17   17
2019                                    0                      24   24
2020                                    0       

## 2. Opening the Zenodo data

The planting (`pd`) and harvest (`hd`) records are plain CSV files. The field outlines are a shapefile, which is a
binary format: a text editor shows garbage, `geopandas` reads it. Both can also be read straight from Zenodo without
saving anything; `gpd.read_file` does not accept the download URL directly, so the zip is read into memory first.

In [3]:
ZEN = SRC / "zenodo"
print(pd.read_csv(ZEN / "farmdb" / "BARC_fields_hd_2019_filt.csv").head(3).to_string(), "\n")

URL = "https://zenodo.org/records/8087301/files/"
fetch = lambda name: io.BytesIO(urllib.request.urlopen(URL + name + "?download=1", timeout=120).read())
outlines_web = gpd.read_file(fetch("barc_field_outlines_for_study.zip"))
print(len(outlines_web), "outlines, EPSG", outlines_web.crs.to_epsg())
print(outlines_web.drop(columns="geometry").head(3).to_string())

                    hd        split  acres               crop     idn
0  2019-11-09 09:00:00  Whole Field   6.45  Soybeans (YSB) /   APU_32
1  2019-11-09 09:00:00  Whole Field   8.80  Soybeans (YSB) /   APU_31
2  2019-11-09 09:00:00  Whole Field  14.22  Soybeans (YSB) /   APU_30 



93 outlines, EPSG 5070
           Comments                id           area   perimeter_  area_ha  area_acre   perm_ft
0  Dairy Production          APU_1-98   33177.212402   958.924447    3.318      8.198  3146.078
1  Dairy Production   CentralFarm_1-1  109511.204834  2639.213100   10.951     27.061  8658.836
2  Dairy Production  CentralFarm_1-11   39339.008057   890.727689    3.934      9.721  2922.335


## 3. Agreement with the official 2021-2023 labels

Re-runs steps 1-4 of `run_all.sh`. In the crosstabs, rows are the farm record and columns are the official label.

| Comparison | Years | Planting records | Harvest records |
|---|---|---|---|
| Zenodo vs official | 2021 only | 12 of 12 | 13 of 13 |
| Farm reports vs official | 2021-2023 | 20 of 20 | 20 of 21 |
| Farm reports vs Zenodo | 2017-2020 | 47 of 47 (planting or harvest label) | |

The one disagreement is `CentralFarm_1-32` in 2023: officially Silage, harvested partly by chopper and partly by
combine.

Limits of this check:
- There is no official label before 2021, so no 2017-2020 label is compared with an official label directly.
- 88 of the 135 Zenodo field-years in 2017-2020 have no farm report, so they are not cross-checked between sources.
- The official labels probably come from the same farm database. Agreement shows the sources are consistent, not that
  they are independently correct.

In [4]:
shp = next((ZEN / "outlines").glob("*.shp"))
run("kraatz_vs_eval.py", ZEN / "farmdb", GPKG, shp, WORK / "zenodo_labels.csv")

2021: Kraatz record vs eval Corn_2021
Corn_2021  Grain  Silage  blank  All
pd                                  
Grain          8       0     22   30
Silage         0       4      5    9
none           6       0     63   69
All           14       4     90  108
Corn_2021  Grain  Silage  blank  All
hd                                  
Grain         10       0     21   31
Silage         0       3      4    7
none           4       1     65   70
All           14       4     90  108
2021 eval-labelled fields where a record differs or is missing:
                  idn      pd     hd harvest_date Corn_2021
19   CentralFarm_1-15     NaN  Grain   2021-10-20     Grain
32   CentralFarm_1-9A     NaN    NaN          NaN     Grain
41   CentralFarm_2-51     NaN  Grain   2021-10-28     Grain
45  CentralFarm_2-68B  Silage    NaN          NaN    Silage
48  CentralFarm_2-72A     NaN    NaN          NaN     Grain
58     EastFarm_5-11A     NaN    NaN          NaN     Grain
91   NorthFarm_ND-5-A     NaN    N

In [5]:
run("pdf2txt.py", SRC / "farm_reports" / "pdf", WORK / "txt", quiet=True)
run("parse_farmmgmt.py", WORK / "txt", WORK / "farm_report_events.csv", quiet=True)
run("mgmt_vs_eval.py", WORK / "farm_report_events.csv", GPKG, WORK / "zenodo_labels.csv",
    WORK / "farm_report_field_years.csv")

harvest records where crop and machine disagree:
                  idn       date     cls  machine        split                                                                                                                  body
22    CentralFarm_1-1 2022-09-09   Grain  Chopper  Whole Field         1-1 | Whole Field | 27.10 | E   Claas 940 Jaguar Chopper | (2022) [RETURNED] | Corn, Field (FLDCRN) / | _____
48   CentralFarm_1-15 2021-10-20   Grain    other  Whole Field  1-15 | Whole Field | 16.88 | E   Claas 350 RC  Round Baler | Corn, Field (FLDCRN) / corn | stalk round bales | _____
97    CentralFarm_1-9 2021-10-20   Grain    other  Whole Field                            1-9 | Whole Field | 32.17 | E   Claas 350 RC  Round Baler | Corn, Field (FLDCRN) / | _____
151   CentralFarm_4-7 2021-11-04  Silage  Combine  Whole Field                               4-7 | Whole Field | 21.29 | E   John Deere S660 Combine | Corn, Silage (CSIL) / | _____

Evaluation labels 2021-2023 vs farm reports (

## 4. A planting record only states the plan

Where a field-year has both a planting and a harvest record, the crop class sometimes differs. Rows are the class at
planting, columns the class at harvest.

In [6]:
zen = pd.read_csv(WORK / "zenodo_labels.csv")
rep = pd.read_csv(WORK / "farm_report_field_years.csv")
zb, rb = zen[zen.pd.notna() & zen.hd.notna()], rep[rep.plant.notna() & rep.harvest.notna()]
print("Zenodo:"); print(pd.crosstab(zb.pd, zb.hd).to_string())
print("\nFarm reports:"); print(pd.crosstab(rb.plant, rb.harvest).to_string())
print("\nField-years where the class changed:")
print(zb[zb.pd != zb.hd][["idn", "year", "pd", "hd", "harvest_date"]].to_string(index=False))
print(rb[rb.plant != rb.harvest][["idn", "year", "plant", "harvest", "harvest_machine", "harvest_date"]].to_string(index=False))

Zenodo:
hd      Grain  Silage
pd                   
Grain      49       0
Silage      1      16

Farm reports:
harvest  Grain  Grain/Silage  Silage
plant                               
Grain       55             0       0
Silage       2             1      22

Field-years where the class changed:
          idn  year     pd    hd harvest_date
EastFarm_5-2A  2019 Silage Grain   2019-09-20
             idn  year  plant      harvest harvest_machine harvest_date
CentralFarm_1-2B  2023 Silage        Grain         Combine   2023-11-15
CentralFarm_1-2D  2023 Silage        Grain         Combine   2023-11-15
CentralFarm_1-32  2023 Silage Grain/Silage Chopper/Combine   2023-09-07


The raw Zenodo rows for `EastFarm_5-2A` in 2019: planted as corn silage in May, harvested as field corn in September.

In [7]:
for kind in ("pd", "hd"):
    d = pd.read_csv(ZEN / "farmdb" / f"BARC_fields_{kind}_2019_filt.csv")
    print(d[(d.idn == "EastFarm_5-2A") & d.crop.str.contains("Corn")].to_string(index=False))

                 pd       split  acres               crop           idn
2019-05-09 08:45:57 Whole Field   8.28 Corn Silage (CSIL) EastFarm_5-2A
                 hd       split  acres                   crop           idn
2019-09-20 09:00:00 Whole Field   8.28 Corn Field (FLDCRN) /  EastFarm_5-2A


How many of the 83 training labels rest on a planting record only:

In [8]:
tr = fy[fy.label_source.str.startswith("farm")].merge(zen[["idn", "year", "pd", "hd"]], on=["idn", "year"], how="left")
tr["basis"] = tr.hd.notna().map({True: "harvest record", False: "planting record only"})
print(pd.crosstab(tr.basis, tr.label, margins=True).to_string())

label                 Grain  Silage  All
basis                                   
harvest record           20       7   27
planting record only     44      12   56
All                      64      19   83


## 5. Does the outline fit the field in that year?

Labels are matched to outlines by farm and field name only, and field boundaries change between years.

**5a. Existing check (`combine_check.py`, part D):** recorded acres against the acres of the evaluation outline.

In [9]:
run("combine_check.py", "--zenodo-dir", ZEN / "farmdb", "--zenodo-labels", WORK / "zenodo_labels.csv",
    "--report-labels", WORK / "farm_report_field_years.csv", "--events", WORK / "farm_report_events.csv",
    "--out", WORK / "extra_labels_outside_2021_2023.csv", tail=14)

              idn  year label  record_acres  outline_acres  ratio
CentralFarm_1-14B  2019 Grain         13.33           6.85   1.95
CentralFarm_1-14C  2020 Grain         11.53           8.59   1.34
 CentralFarm_1-7B  2017 Grain          2.76          26.96   0.10
 CentralFarm_1-9A  2017 Grain          1.81          32.61   0.06
CentralFarm_2-71A  2017 Grain          3.63          40.70   0.09
CentralFarm_2-72A  2017 Grain          7.22          41.00   0.18
   EastFarm_5-45D  2015 Grain          8.52          17.30   0.49
    EastFarm_6-60  2019 Grain         25.20          18.94   1.33
    EastFarm_6-60  2020 Grain         25.28          18.94   1.33
    EastFarm_6-64  2019 Grain         22.67          29.89   0.76
    EastFarm_6-64  2020 Grain         22.07          29.89   0.74

wrote ..\labels_src\rerun\extra_labels_outside_2021_2023.csv


**5b. New check: overlap with a second outline.** Acres alone cannot show where a field is. For each training
field-year this compares the outline actually stored in `field_years.csv` with the evaluation outline and the Zenodo
study outline of the same field name (IoU = intersection area / union area), and with the recorded acres.

Of the 83 training field-years, 69 use the evaluation outline and 14 use the Zenodo outline.

In [10]:
ev = gpd.read_file(GPKG).to_crs(32618)
ev["idn"] = ev["FARM"].str.replace(" ", "") + "_" + ev["FIELD"].astype(str)
zo = gpd.read_file(shp).to_crs(32618)
zo["idn"] = zo["id"].str.strip()
eg, zg = ev.dissolve("idn").geometry, zo.dissolve("idn").geometry
iou = lambda a, b: a.intersection(b).area / a.union(b).area

t = fy[fy.label_source.str.startswith("farm")].copy()
t["geom"] = t.wkt.map(wkt.loads)
t["iou_eval"] = [iou(g, eg[i]) if i in eg.index else None for g, i in zip(t.geom, t.idn)]
t["iou_zenodo"] = [iou(g, zg[i]) if i in zg.index else None for g, i in zip(t.geom, t.idn)]
t["outline_used"] = t.iou_eval.notna().map({True: "evaluation", False: "Zenodo"})

acres = []
for f in sorted((ZEN / "farmdb").glob("BARC_fields_*_filt.csv")):
    d = pd.read_csv(f)
    acres.append(d[d.split.str.strip() == "Whole Field"].assign(year=int(f.name.split("_")[3]))[["idn", "year", "acres"]])
t = t.join(pd.concat(acres).groupby(["idn", "year"]).acres.max().rename("record_acres"), on=["idn", "year"])
t["outline_acres"] = t.geom.map(lambda g: g.area / 4046.86)
t["ratio"] = t.record_acres / t.outline_acres


def status(r):
    ok = 0.8 <= r.ratio <= 1.25 if pd.notna(r.ratio) else None
    if r.outline_used == "Zenodo":
        return "1 Zenodo outline, acres match"
    if pd.notna(r.iou_zenodo) and r.iou_zenodo >= 0.9 and ok is not False:
        return "2 evaluation outline, IoU with Zenodo outline >= 0.9"
    if pd.isna(r.iou_zenodo) and ok:
        return "3 evaluation outline, no Zenodo outline, acres match"
    if ok:
        return "4 evaluation outline, IoU < 0.9 but acres match"
    return "5 does not fit"


t["status"] = t.apply(status, axis=1)
print(t.outline_used.value_counts().to_string(), "\n")
print(pd.crosstab(t.status, t.label, margins=True).to_string())
print("\nField-years that do not fit:")
print(t[t.status.str.startswith("5")][["idn", "year", "label", "record_acres", "outline_acres", "ratio", "iou_zenodo"]]
      .sort_values(["idn", "year"]).round(2).to_string(index=False))

outline_used
evaluation    69
Zenodo        14 

label                                                 Grain  Silage  All
status                                                                  
1 Zenodo outline, acres match                            13       1   14
2 evaluation outline, IoU with Zenodo outline >= 0.9     27      10   37
3 evaluation outline, no Zenodo outline, acres match     11       7   18
4 evaluation outline, IoU < 0.9 but acres match           3       1    4
5 does not fit                                           10       0   10
All                                                      64      19   83

Field-years that do not fit:
              idn  year label  record_acres  outline_acres  ratio  iou_zenodo
CentralFarm_1-14B  2019 Grain         13.33           6.81   1.96        0.28
CentralFarm_1-14C  2020 Grain         11.53           8.64   1.33         NaN
 CentralFarm_1-7B  2017 Grain          2.76          26.95   0.10         NaN
 CentralFarm_1-9A  2017 G

Reading the result:
- The 10 field-years that do not fit are all Grain, and are the same 10 that `combine_check.py` flags.
- In the four 2017 cases the record covers 2-7 acres of a 27-41 acre outline, so most of the outline is other ground.
- `CentralFarm_1-14B` and `EastFarm_6-60` fit the evaluation outline in 2018 and the larger Zenodo outline from 2019:
  the field was enlarged. `CentralFarm_1-14B` 2018 (Silage) is therefore kept.
- Group 3 (18 field-years) is confirmed by acres only, which cannot rule out a shifted outline of the same size.
- The Zenodo outlines are one set, not one per year, so they do not prove the boundary of any single year.

## 6. Compared with the code on `main`, and what is still open

| | Code on `main` | This branch |
|---|---|---|
| Field-years flagged for outline fit | 10 in the training set (by acres) | the same 10 (acres and overlap) |
| Are they removed from training? | No, all 83 are used | optional: `prithvi_train.py --exclude outline-misfit` |
| Outline position checked | No, acres only | Yes, for the 46 with both outlines |
| The 14 on Zenodo outlines | not covered by part D | acres match in all 14 |
| Planting-only Silage (12) | flagged as least certain | same |

Next step, not yet run (needs the GPU machine): retrain with (a) `--exclude outline-misfit` and
(b) `--exclude outline-misfit planting-only-silage`, and compare with the current result. Removing the 10 leaves
54 Grain and 19 Silage farm-record field-years. The commands are in `classifier/prithvi/README.md`.

`outline_check.py` runs the check of section 5b as step 8 of `run_all.sh` and writes
`results/training_label_checks.csv`.

Other caveats that stay: one farm only; 5 official Grain field-years look like soybeans in the USDA Cropland Data
Layer (see `classifier/labels/README.md`).

In [11]:
chk = pd.read_csv(LABELS / "results" / "training_label_checks.csv")
mine = set(map(tuple, t[t.status.str.startswith("5")][["field_id", "year"]].to_numpy()))
theirs = set(map(tuple, chk[chk.outline_fit == "misfit"][["field_id", "year"]].to_numpy()))
print("misfit field-years here:", len(mine), "| in training_label_checks.csv:", len(theirs), "| same set:", mine == theirs)
print(pd.crosstab(fy.label_source, fy.outline_fit).to_string())

misfit field-years here: 10 | in training_label_checks.csv: 10 | same set: True
outline_fit                  misfit  official  ok
label_source                                     
evaluation labels 2021-2023       0        44   0
farm records 2017-2020           10         0  73
